# Hotel Reviews — NLP enrichment (Google Colab / GPU)

Runs the same free, local sentiment + aspect pipeline as `src/nlp_pipeline.py`, adapted for a
Colab GPU runtime:

- **fp16** inference (roughly 2x faster on GPU, negligible accuracy loss for this task)
- **larger batches** tuned for a T4 (Colab's free-tier GPU, 16GB)
- **checkpointing every 50k reviews** to Google Drive, so a Colab disconnect never loses more
  than one chunk — re-running the "full run" cell automatically skips finished chunks

**Before running:** `Runtime -> Change runtime type -> T4 GPU` (or better, if you have Colab Pro).

**You need in your Google Drive:** the raw `Hotel_Reviews.csv` (~230MB) uploaded somewhere —
default path assumed below is `MyDrive/hotelReviews/data/Hotel_Reviews.csv`. Adjust `DATA_PATH`
in the config cell if you put it elsewhere.


## 1. Check the GPU

In [ ]:
import torch

if torch.cuda.is_available():
    print("GPU OK:", torch.cuda.get_device_name(0))
    print("This determines your speed — if this prints CPU only, go to")
    print("Runtime -> Change runtime type -> T4 GPU, then Runtime -> Restart session.")
else:
    print("!! No GPU detected. Go to Runtime -> Change runtime type -> T4 GPU, then restart.")


## 2. Install extra dependencies

(`torch`, `pandas`, `tqdm`, `pyarrow` already ship with Colab.)

In [ ]:
!pip install -q "transformers>=4.40" sentencepiece protobuf


## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 4. Configure paths

Edit `DATA_PATH` if your CSV lives somewhere else in Drive. Everything else (checkpoints, final
output) is created automatically under `OUTPUT_DIR`.


In [ ]:
from pathlib import Path

DATA_PATH = Path("/content/drive/MyDrive/hotelReviews/data/Hotel_Reviews.csv")
OUTPUT_DIR = Path("/content/drive/MyDrive/hotelReviews/outputs")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_PATH.exists(), f"CSV not found at {DATA_PATH} — upload it to Drive or fix the path."
print("Data file OK:", DATA_PATH, f"({DATA_PATH.stat().st_size / 1e6:.0f} MB)")


## 5. Text cleaning (inlined from `src/text_cleaning.py`)

Same logic as the local pipeline: strip placeholders (`"No Negative"` / `"No Positive"`), repair
the contractions the scraper's punctuation-stripping broke (`"don t"` -> `"don't"`), and join both
review sides into one text per review.


In [ ]:
import re

NEGATIVE_PLACEHOLDER = "No Negative"
POSITIVE_PLACEHOLDER = "No Positive"
MULTISPACE = re.compile(r"\s{2,}")

CONTRACTION_FIXES: dict[str, str] = {
    r"\bcan not\b": "cannot",
    r"\b(do|does|did|is|was|are|were|has|have|had|would|should|could|"
    r"wo|ca|ai|must|need|might)n t\b": r"\1n\'t",
    r"\bit s\b": "it\'s",
    r"\bthat s\b": "that\'s",
    r"\bthere s\b": "there\'s",
    r"\bhe s\b": "he\'s",
    r"\bshe s\b": "she\'s",
    r"\bwhat s\b": "what\'s",
    r"\blet s\b": "let\'s",
    r"\bi m\b": "I\'m",
    r"\b(i|we|they|you) ve\b": r"\1\'ve",
    r"\b(i|we|they|you|it|he|she) ll\b": r"\1\'ll",
    r"\b(i|we|they|you|he|she|it) d\b": r"\1\'d",
    r"\b(we|they|you) re\b": r"\1\'re",
}
_COMPILED_FIXES = [(re.compile(p, re.IGNORECASE), r) for p, r in CONTRACTION_FIXES.items()]


def fix_broken_contractions(text: str) -> str:
    for pattern, repl in _COMPILED_FIXES:
        text = pattern.sub(repl, text)
    return text


def normalize_text(text: str) -> str:
    text = MULTISPACE.sub(" ", str(text).strip())
    return fix_broken_contractions(text)


def clean_review_side(series, placeholder: str):
    s = series.astype(str).str.strip()
    s = s.mask(s == placeholder, "")
    return s.map(lambda t: normalize_text(t) if t else "")


def combined_clean_text(df):
    pos = clean_review_side(df["Positive_Review"], POSITIVE_PLACEHOLDER)
    neg = clean_review_side(df["Negative_Review"], NEGATIVE_PLACEHOLDER)
    return (pos + " . " + neg).str.strip(" .").str.replace(r"\s+", " ", regex=True)


## 6. Models (fp16, tuned batch sizes)

`torch_dtype=torch.float16` is the main speed lever on GPU. Batch sizes below are safe starting
points for a T4 (16GB) — raise them if you have a bigger GPU (A100 in Colab Pro) and watch for
`CUDA out of memory`, in which case lower them back down.


In [ ]:
import re
import torch
from transformers import pipeline

OVERALL_MODEL = "nlptown/bert-base-multilingual-uncased-sentiment"
ABSA_MODEL = "yangheng/deberta-v3-base-absa-v1.1"

OVERALL_BATCH_SIZE = 256   # BERT-base is light -> can push batch size higher
ABSA_BATCH_SIZE = 128      # DeBERTa-v3-base is heavier -> smaller batch

DEVICE = 0 if torch.cuda.is_available() else -1
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

# Hotel-relevant aspects and the keywords that "unlock" ABSA scoring for them.
# Keys become column names; extend freely with business-specific aspects.
#
# Matching is WORD-BOUNDARY SAFE (see ASPECT_PATTERNS below): a short keyword
# like "bar" only matches the standalone word "bar"/"bars", never a substring
# inside an unrelated word like "barefoot" or "barely". Because of that, common
# inflected forms are listed explicitly instead of relying on substring
# matching to catch them (e.g. "clean" no longer implicitly covers "cleaning" —
# both are listed).
# Hotel-relevant aspects. Each aspect has a small set of closed-vocabulary
# SUB-TAGS — this is what makes problems COMPARABLE and COUNTABLE across
# reviews and hotels (e.g. "37 reviews this quarter tagged water_quality"),
# unlike a free-text evidence snippet, which is unique to each review and
# cannot be aggregated. Matching is WORD-BOUNDARY SAFE (see *_PATTERNS below):
# a short keyword like "bar" only matches the standalone word, never a
# substring inside an unrelated word like "barefoot".
ASPECT_LEXICON: dict[str, dict[str, tuple[str, ...]]] = {
    "cleanliness": {
        "smell": ("smell", "smells", "smelly", "smelled", "odour", "odor",
                  "stink", "stinky", "musty"),
        "dirt_dust": ("dirty", "dirtier", "dust", "dusty", "stain", "stains",
                      "stained", "filthy", "grime", "grimy"),
        "mold": ("mould", "mouldy", "mold", "moldy"),
        "hygiene": ("hygiene", "hygienic", "unhygienic"),
        "housekeeping_general": ("clean", "cleaning", "cleaned", "cleaner",
                                  "cleanliness", "tidy", "untidy", "spotless"),
    },
    "staff": {
        "rudeness": ("rude", "unfriendly", "impolite"),
        "helpfulness": ("helpful", "unhelpful", "welcoming", "friendly", "polite"),
        "checkin_checkout": ("reception", "receptionist", "check in",
                              "check out", "checkin", "checkout"),
        "management": ("manager", "managers"),
        "staff_general": ("staff", "employee", "employees", "service"),
    },
    "location": {
        "distance_center": ("central", "centre", "center", "near", "nearby",
                             "close", "distance", "walk", "walking"),
        "transport_access": ("metro", "station", "bus stop", "tram"),
        "surroundings": ("area", "neighbourhood", "neighborhood", "surroundings"),
        "location_general": ("location", "located"),
    },
    "comfort": {
        "bed_mattress": ("bed", "beds", "mattress", "pillow", "pillows"),
        "room_size": ("spacious", "cramped", "room size"),
        "sleep_quality": ("sleep", "sleeping", "comfortable", "uncomfortable",
                           "cozy", "cosy", "comfy"),
    },
    "food": {
        "breakfast": ("breakfast",),
        "restaurant_service": ("restaurant", "meal", "meals", "dinner",
                                "lunch", "buffet"),
        "drinks_bar": ("bar", "bars", "drink", "drinks", "coffee"),
        "food_general": ("food",),
    },
    "value": {
        "price_too_high": ("expensive", "overpriced", "pricey"),
        "hidden_fees": ("hidden fee", "hidden fees", "extra charge",
                         "extra charges", "surcharge"),
        "good_value": ("cheap", "affordable", "worth", "value for money"),
        "value_general": ("price", "prices", "cost", "costs", "money", "value"),
    },
    "noise": {
        "external_noise": ("street noise", "traffic noise", "outside noise"),
        "internal_noise": ("thin walls", "neighbours", "neighbors", "noisy room"),
        "noise_general": ("noise", "noisy", "loud", "quiet", "soundproof", "silent"),
    },
    "facilities": {
        "wifi": ("wifi", "internet", "wi-fi"),
        "water_quality": ("water pressure", "hot water", "cold water",
                           "water quality", "no water", "water"),
        "air_conditioning": ("air conditioning", "ac unit", "aircon"),
        "pool_gym_spa": ("pool", "gym", "spa", "spas"),
        "parking": ("parking",),
        "elevator": ("elevator", "elevators", "lift", "lifts"),
        "bathroom": ("shower", "showers", "bathroom", "bathrooms"),
        "heating": ("heating",),
    },
}


def _compile_pattern(keywords: tuple[str, ...]) -> "re.Pattern[str]":
    """Word-boundary-safe regex for a list of keywords/phrases (see module docstring)."""
    parts = [
        re.escape(kw) if " " in kw else rf"\b{re.escape(kw)}\b"
        for kw in keywords
    ]
    return re.compile("|".join(parts))


# One pattern per sub-tag, and one merged pattern per aspect (union of its
# sub-tags) reused for the aspect-level gate and the evidence snippet.
SUBTAG_PATTERNS: dict[str, dict[str, "re.Pattern[str]"]] = {
    aspect: {subtag: _compile_pattern(keywords) for subtag, keywords in subtags.items()}
    for aspect, subtags in ASPECT_LEXICON.items()
}
ASPECT_PATTERNS: dict[str, "re.Pattern[str]"] = {
    aspect: _compile_pattern(tuple(kw for keywords in subtags.values() for kw in keywords))
    for aspect, subtags in ASPECT_LEXICON.items()
}

print("Loading overall-sentiment model...")
overall_clf = pipeline("sentiment-analysis", model=OVERALL_MODEL, device=DEVICE, torch_dtype=DTYPE)
print("Loading aspect (ABSA) model...")
absa_clf = pipeline("text-classification", model=ABSA_MODEL, device=DEVICE, torch_dtype=DTYPE)
print("Both models loaded on", "GPU (fp16)" if DEVICE == 0 else "CPU (fp32 — this will be slow)")


## 7. Scoring functions (deduplicated + batched)

Per mentioned aspect, produces **two columns**, never one categorical label:

- `aspect_<name>_score` -- a **0-10 float** (0 = most negative, 10 = most positive),
  remapped from the ABSA classifier's own confidence. `NaN` when the aspect is never
  mentioned (not `5` or `0` -- NaN keeps "not mentioned" distinct from "scored neutral").
- `aspect_<name>_evidence` -- a **verbatim excerpt** from the review around the matched
  keyword(s). This is plain word-slicing, not a model generation step, so it is
  structurally incapable of inventing text that isn't in the source review.


In [ ]:
from tqdm.auto import tqdm

EVIDENCE_MAX_SNIPPETS = 2  # cap snippets per aspect so the column stays readable


def _classify_batched(clf, inputs: list, batch_size: int, desc: str) -> list[dict]:
    out: list[dict] = []
    for i in tqdm(range(0, len(inputs), batch_size), desc=desc):
        out.extend(clf(inputs[i : i + batch_size], truncation=True, batch_size=batch_size))
    return out


def score_overall_sentiment(texts):
    unique = texts[texts.str.len() > 0].drop_duplicates()
    preds = _classify_batched(overall_clf, unique.tolist(), OVERALL_BATCH_SIZE, "overall")
    stars = {t: int(p["label"].split()[0]) for t, p in zip(unique, preds)}
    return texts.map(lambda t: stars.get(t))




def detect_aspects(text_lower: str) -> list[str]:
    """Return the aspects whose (merged) keyword pattern matches in a lowercased review."""
    return [aspect for aspect, pattern in ASPECT_PATTERNS.items() if pattern.search(text_lower)]


def detect_subtags(aspect: str, text_lower: str) -> list[str]:
    """Return the closed-vocabulary sub-tags of `aspect` matched in a lowercased review.

    This is the countable/comparable signal — e.g. facilities -> ["water_quality"]
    can be aggregated across every review and hotel, unlike a free-text evidence
    snippet, which is unique per review.
    """
    return [subtag for subtag, pattern in SUBTAG_PATTERNS[aspect].items() if pattern.search(text_lower)]


# Characters of context kept on each side of a keyword match in the evidence
# snippet, and how many non-overlapping snippets to keep per aspect.
EVIDENCE_CONTEXT_CHARS = 45
EVIDENCE_MAX_SNIPPETS = 2


def _score_from_label(label: str, confidence: float) -> float:
    """Map an ABSA label + confidence to a 0-10 scale (0 = most negative, 10 = most positive).

    Deterministic remap of the classifier own confidence -- no separate model,
    nothing invented, just a rescale of a number the model already produced.
    """
    label = label.lower()
    if label == "positive":
        strength = confidence
    elif label == "negative":
        strength = -confidence
    else:  # neutral
        strength = 0.0
    return round(5.0 + 5.0 * strength, 2)


def _extract_evidence(text: str, aspect: str) -> str:
    """Return verbatim context snippet(s) around each aspect keyword hit.

    Audit trail only — not the comparable signal; use detect_subtags for
    anything that needs to be counted or aggregated.
    """
    pattern = ASPECT_PATTERNS[aspect]
    lowered = text.lower()
    snippets: list[str] = []
    last_end = -1
    for m in pattern.finditer(lowered):
        if m.start() < last_end:
            continue
        lo = max(0, m.start() - EVIDENCE_CONTEXT_CHARS)
        hi = min(len(text), m.end() + EVIDENCE_CONTEXT_CHARS)
        snippets.append(text[lo:hi].strip())
        last_end = hi
        if len(snippets) >= EVIDENCE_MAX_SNIPPETS:
            break
    return " (...) ".join(snippets)


def score_aspects(texts):
    """Per mentioned aspect: a 0-10 score, closed-vocabulary sub-tags (comparable
    across reviews), and a verbatim evidence snippet (audit trail only).

    Three columns per aspect — aspect_<name>_score (float, NaN when not
    mentioned), aspect_<name>_subtags (";"-joined closed tags, "" when not
    mentioned), aspect_<name>_evidence (verbatim excerpt, "" when not mentioned).
    """
    lowered = texts.str.lower()
    mentioned = lowered.map(detect_aspects)

    jobs: set[tuple[str, str]] = set()
    for text, aspects in zip(texts, mentioned):
        for aspect in aspects:
            jobs.add((text, aspect))
    job_list = sorted(jobs)

    inputs = [{"text": t, "text_pair": a} for t, a in job_list]
    preds = _classify_batched(absa_clf, inputs, ABSA_BATCH_SIZE, "aspects")

    score_cache: dict[tuple[str, str], float] = {}
    subtag_cache: dict[tuple[str, str], str] = {}
    evidence_cache: dict[tuple[str, str], str] = {}
    for (text, aspect), pred in zip(job_list, preds):
        score_cache[(text, aspect)] = _score_from_label(pred["label"], pred["score"])
        subtag_cache[(text, aspect)] = ";".join(detect_subtags(aspect, text.lower()))
        evidence_cache[(text, aspect)] = _extract_evidence(text, aspect)

    import pandas as pd
    result = pd.DataFrame(index=texts.index)
    for aspect in ASPECT_LEXICON:
        result[f"aspect_{aspect}_score"] = [
            score_cache.get((text, aspect)) if aspect in aspects else None
            for text, aspects in zip(texts, mentioned)
        ]
        result[f"aspect_{aspect}_subtags"] = [
            subtag_cache.get((text, aspect), "") if aspect in aspects else ""
            for text, aspects in zip(texts, mentioned)
        ]
        result[f"aspect_{aspect}_evidence"] = [
            evidence_cache.get((text, aspect), "") if aspect in aspects else ""
            for text, aspects in zip(texts, mentioned)
        ]
    result["n_aspects_mentioned"] = mentioned.map(len).values
    return result


## 8. Chunked, resumable run

Processes `Hotel_Reviews.csv` in chunks of `CHUNK_SIZE` rows. Each finished chunk is saved to
`CHECKPOINT_DIR` immediately — if Colab disconnects, just re-run this cell: already-saved chunks
are skipped automatically.


In [ ]:
import pandas as pd

CHUNK_SIZE = 50_000

COLS = ["Hotel_Name", "Review_Date", "Reviewer_Score",
        "Reviewer_Nationality", "Positive_Review", "Negative_Review", "Tags"]


def process_chunk(df_chunk: pd.DataFrame, start_idx: int) -> pd.DataFrame:
    df_chunk = df_chunk.reset_index(drop=True)
    df_chunk["review_id"] = range(start_idx, start_idx + len(df_chunk))
    df_chunk["quarter"] = (
        pd.to_datetime(df_chunk["Review_Date"], format="%m/%d/%Y", errors="coerce")
        .dt.to_period("Q").astype(str)
    )
    text = combined_clean_text(df_chunk)
    df_chunk["overall_stars"] = score_overall_sentiment(text)
    aspects = score_aspects(text)
    return pd.concat(
        [df_chunk[["review_id", "Hotel_Name", "Review_Date", "quarter",
                   "Reviewer_Nationality", "Reviewer_Score", "Tags", "overall_stars"]],
         aspects],
        axis=1,
    )


def run_full(sample: int | None = None):
    # Each run "shape" gets its OWN checkpoint folder (e.g. checkpoints/sample_500/
    # vs checkpoints/full/), so a smoke test and the full run can never read or
    # skip each other's chunks by mistake, even if chunk indices collide (both
    # start at chunk_0000.parquet).
    run_tag = f"sample_{sample}" if sample else "full"
    run_ckpt_dir = CHECKPOINT_DIR / run_tag
    run_ckpt_dir.mkdir(parents=True, exist_ok=True)

    n_rows = sample or sum(1 for _ in open(DATA_PATH, encoding="utf-8", errors="ignore")) - 1
    n_chunks = (n_rows + CHUNK_SIZE - 1) // CHUNK_SIZE
    print(f"[{run_tag}] processing {n_rows:,} rows in {n_chunks} chunk(s) of {CHUNK_SIZE:,}")
    print(f"[{run_tag}] checkpoints -> {run_ckpt_dir}")

    reader = pd.read_csv(DATA_PATH, usecols=COLS, low_memory=False, chunksize=CHUNK_SIZE,
                          nrows=sample)
    for chunk_idx, df_chunk in enumerate(reader):
        ckpt_path = run_ckpt_dir / f"chunk_{chunk_idx:04d}.parquet"
        if ckpt_path.exists():
            print(f"[{chunk_idx+1}/{n_chunks}] already done, skipping ({ckpt_path.name})")
            continue
        print(f"[{chunk_idx+1}/{n_chunks}] processing rows "
              f"{chunk_idx*CHUNK_SIZE:,}-{chunk_idx*CHUNK_SIZE+len(df_chunk):,} ...")
        enriched = process_chunk(df_chunk, start_idx=chunk_idx * CHUNK_SIZE)
        enriched.to_parquet(ckpt_path, index=False)
        print(f"  saved {ckpt_path.name} ({len(enriched):,} rows)")

    parts = sorted(run_ckpt_dir.glob("chunk_*.parquet"))
    out_path = OUTPUT_DIR / (
        "reviews_enriched_sample.parquet" if sample else "reviews_enriched.parquet"
    )
    if len(parts) == n_chunks:
        full = pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True)
        full.to_parquet(out_path, index=False)
        print(f"\nDONE [{run_tag}] — {len(full):,} enriched reviews -> {out_path}")
        return full
    else:
        print(f"\n[{run_tag}] {len(parts)}/{n_chunks} chunks done — re-run this cell to "
              "continue (finished chunks are skipped).")
        return None


## 9. Smoke test first (500 rows, ~1 minute)

Always run this before the full job.

In [ ]:
run_full(sample=500)

## 9b. Export the sample as CSV (optional, easy to inspect)


In [ ]:
sample_output_path = OUTPUT_DIR / "reviews_enriched_sample.parquet"
if sample_output_path.exists():
    df_enriched_sample = pd.read_parquet(sample_output_path)
    csv_output_path = OUTPUT_DIR / "reviews_enriched_sample.csv"
    df_enriched_sample.to_csv(csv_output_path, index=False)
    print(f"Sample enriched reviews saved to: {csv_output_path}")
else:
    print(f"Parquet file not found at {sample_output_path}. Run the previous cell first.")


## 10. Full run (all ~515,738 reviews)

Estimated time on a Colab T4 with fp16: roughly **1-3 hours**, depending on GPU contention on
the free tier. Safe to stop and re-run this cell at any point — finished chunks persist in Drive
and are skipped.


In [ ]:
run_full(sample=None)

## 10b. Export the full result as CSV (optional)

The full dataset is large (~515k rows) — this can take a minute and the
resulting CSV will be bigger than the parquet file (parquet is compressed,
CSV is plain text). Skip this cell if you only need the parquet for the
next phase.


In [ ]:
full_output_path = OUTPUT_DIR / "reviews_enriched.parquet"
if full_output_path.exists():
    df_enriched_full = pd.read_parquet(full_output_path)
    csv_output_path = OUTPUT_DIR / "reviews_enriched.csv"
    df_enriched_full.to_csv(csv_output_path, index=False)
    print(f"Full enriched dataset saved to: {csv_output_path}")
else:
    print(f"Parquet file not found at {full_output_path}. Run the previous cell first.")


## 11. Dashboard data prep (mirrors the website's mock data)

Turns the enriched reviews into the same aggregated shapes the TrueStay dashboard
uses (`web/src/data/mockData.js`): category scores per hotel, a leaderboard,
quarterly trend, comment samples, and a competitor gap matrix. Uses the full
dataset if it finished running, otherwise falls back to the smoke-test sample.

Two honest gaps vs. the mocked website — read before wiring this to real data:

- The mock's "Monthly score trend" is monthly; this only has **quarterly**
  granularity (matches the hotel x quarter panel decided for the project).
- The mock's Quadrant page (price vs. quality) has no equivalent here — the
  dataset has no price/rate column.


In [ ]:
import json

WEBSITE_ASPECTS = ["food", "comfort", "cleanliness", "staff", "location"]
NEGATIVE_THRESHOLD = 5.0  # same 0-10 midpoint _score_from_label uses


def dp_title(aspect: str) -> str:
    return "Value for Money" if aspect == "value" else aspect.capitalize()


def dp_category_scores_by_hotel(df):
    rows = []
    for hotel, g in df.groupby("Hotel_Name"):
        row = {"Hotel_Name": hotel, "n_reviews": len(g)}
        for aspect in ASPECT_LEXICON:
            col = g[f"aspect_{aspect}_score"]
            row[f"{aspect}_score"] = round(col.mean(), 2) if col.notna().any() else None
            row[f"{aspect}_mentions"] = int(col.notna().sum())
        rows.append(row)
    return pd.DataFrame(rows)


def dp_leaderboard(df):
    board = (
        df.groupby("Hotel_Name")["Reviewer_Score"]
        .agg(overall_score="mean", n_reviews="size")
        .round({"overall_score": 2})
        .sort_values("overall_score", ascending=False)
        .reset_index()
    )
    board.insert(0, "rank", range(1, len(board) + 1))
    return board


def dp_quarterly_trend(df, hotel_name):
    hotel_df = df[df["Hotel_Name"] == hotel_name]
    overall = hotel_df.groupby("quarter")["Reviewer_Score"].mean().round(2)
    trend = pd.DataFrame({"overall": overall})
    for aspect in WEBSITE_ASPECTS:
        trend[aspect] = hotel_df.groupby("quarter")[f"aspect_{aspect}_score"].mean().round(2)
    return trend.reset_index()


def dp_dimension_comments(df, hotel_name):
    hotel_df = df[df["Hotel_Name"] == hotel_name]
    rows = []
    for aspect in WEBSITE_ASPECTS:
        col = hotel_df[f"aspect_{aspect}_score"]
        rows.append({
            "name": dp_title(aspect),
            "total": int(col.notna().sum()),
            "negative": int((col < NEGATIVE_THRESHOLD).sum()),
        })
    return pd.DataFrame(rows)


def dp_category_comments(df, hotel_name, per_side=4):
    hotel_df = df[df["Hotel_Name"] == hotel_name]
    result = {}
    for aspect in WEBSITE_ASPECTS:
        score_col, evidence_col = f"aspect_{aspect}_score", f"aspect_{aspect}_evidence"
        mentioned = hotel_df[hotel_df[score_col].notna() & (hotel_df[evidence_col] != "")]
        picked = []
        for sentiment, mask in [
            ("positive", mentioned[score_col] >= NEGATIVE_THRESHOLD),
            ("negative", mentioned[score_col] < NEGATIVE_THRESHOLD),
        ]:
            for _, r in mentioned[mask].head(per_side).iterrows():
                picked.append({
                    "sentiment": sentiment,
                    "text": r[evidence_col],
                    "nationality": str(r["Reviewer_Nationality"]).strip(),
                    "date": r["Review_Date"],
                })
        result[dp_title(aspect)] = picked
    return result


def dp_competitor_gaps(scores_by_hotel, focus_hotel):
    focus_row = scores_by_hotel[scores_by_hotel["Hotel_Name"] == focus_hotel].iloc[0]
    rows = []
    for _, comp in scores_by_hotel[scores_by_hotel["Hotel_Name"] != focus_hotel].iterrows():
        row = {"competitor": comp["Hotel_Name"]}
        for aspect in WEBSITE_ASPECTS:
            fv, cv = focus_row[f"{aspect}_score"], comp[f"{aspect}_score"]
            row[dp_title(aspect)] = round(fv - cv, 2) if pd.notna(fv) and pd.notna(cv) else None
        rows.append(row)
    return pd.DataFrame(rows)


def dp_vulnerability_table(df, scores_by_hotel, focus_hotel):
    entries = []
    for _, comp in scores_by_hotel[scores_by_hotel["Hotel_Name"] != focus_hotel].iterrows():
        aspect_scores = {a: comp[f"{a}_score"] for a in WEBSITE_ASPECTS if pd.notna(comp[f"{a}_score"])}
        if not aspect_scores:
            continue
        weakest = min(aspect_scores, key=aspect_scores.get)
        comp_df = df[df["Hotel_Name"] == comp["Hotel_Name"]]
        score_col, evidence_col = f"aspect_{weakest}_score", f"aspect_{weakest}_evidence"
        negative = comp_df[(comp_df[score_col] < NEGATIVE_THRESHOLD) & (comp_df[evidence_col] != "")]
        if negative.empty:
            continue
        entries.append({
            "competitor": comp["Hotel_Name"],
            "category": dp_title(weakest),
            "review": negative.iloc[0][evidence_col],
        })
    return entries


# Prefer the full dataset if it finished; fall back to the smoke-test sample.
_enriched_path = OUTPUT_DIR / "reviews_enriched.parquet"
if not _enriched_path.exists():
    _enriched_path = OUTPUT_DIR / "reviews_enriched_sample.parquet"
print(f"Using {_enriched_path.name}")

dashboard_df = pd.read_parquet(_enriched_path)
focus_hotel = dashboard_df["Hotel_Name"].value_counts().idxmax()
print(f"{len(dashboard_df)} reviews, {dashboard_df['Hotel_Name'].nunique()} hotels — "
      f"focus hotel: {focus_hotel}")

dp_scores = dp_category_scores_by_hotel(dashboard_df)
dp_board = dp_leaderboard(dashboard_df)
dp_trend = dp_quarterly_trend(dashboard_df, focus_hotel)
dp_dim_comments = dp_dimension_comments(dashboard_df, focus_hotel)
dp_comments = dp_category_comments(dashboard_df, focus_hotel)
dp_gaps = dp_competitor_gaps(dp_scores, focus_hotel)
dp_vulns = dp_vulnerability_table(dashboard_df, dp_scores, focus_hotel)

print("\nCategory scores by hotel:")
print(dp_scores.to_string(index=False))
print("\nLeaderboard:")
print(dp_board.to_string(index=False))

dashboard_data = {
    "focus_hotel": focus_hotel,
    "category_scores_by_hotel": dp_scores.to_dict(orient="records"),
    "leaderboard": dp_board.to_dict(orient="records"),
    "quarterly_trend": dp_trend.to_dict(orient="records"),
    "dimension_comments": dp_dim_comments.to_dict(orient="records"),
    "category_comments": dp_comments,
    "competitor_gaps": dp_gaps.to_dict(orient="records"),
    "vulnerabilities": dp_vulns,
}
dashboard_json_path = OUTPUT_DIR / "dashboard_data.json"
dashboard_json_path.write_text(json.dumps(dashboard_data, indent=2, default=str), encoding="utf-8")
print(f"\nSaved -> {dashboard_json_path}")


## 12. Dashboard charts

Renders inline (Colab shows matplotlib figures automatically) and also saves
each one to Drive under `outputs/figures/`.


In [ ]:
import matplotlib.pyplot as plt

CATEGORY_COLORS = {
    "Food": "#f97316", "Comfort": "#ef4444", "Cleanliness": "#06b6d4",
    "Staff": "#8b5cf6", "Location": "#22c55e",
}
CHART_ASPECTS = ["Food", "Comfort", "Cleanliness", "Staff", "Location"]

FIGURES_DIR = OUTPUT_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def _aspect_key(display_name):
    return "value" if display_name == "Value for Money" else display_name.lower()


# --- Category scores ---
_row = next(r for r in dashboard_data["category_scores_by_hotel"] if r["Hotel_Name"] == focus_hotel)
_scores = [_row[f"{_aspect_key(a)}_score"] for a in CHART_ASPECTS]

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(CHART_ASPECTS, _scores, color=[CATEGORY_COLORS[a] for a in CHART_ASPECTS])
ax.axhline(7.0, color="#94a3b8", linestyle="--", linewidth=1, label="Stable threshold (7.0)")
ax.set_ylim(0, 10)
ax.set_ylabel("Score (0-10)")
ax.set_title(f"Category scores — {focus_hotel}")
ax.legend()
for bar, score in zip(bars, _scores):
    ax.text(bar.get_x() + bar.get_width() / 2, score + 0.15, f"{score:.1f}", ha="center", fontsize=10)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "dashboard_category_scores.png", dpi=110)
plt.show()

# --- Quarterly trend ---
_quarters = [r["quarter"] for r in dashboard_data["quarterly_trend"]]
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(_quarters, [r["overall"] for r in dashboard_data["quarterly_trend"]],
        color="#2563eb", linewidth=2.5, marker="o", label="Overall")
for a in CHART_ASPECTS:
    key = _aspect_key(a)
    if all(key in r for r in dashboard_data["quarterly_trend"]):
        ax.plot(_quarters, [r.get(key) for r in dashboard_data["quarterly_trend"]],
                 color=CATEGORY_COLORS[a], linewidth=1.3, alpha=0.7, label=a)
ax.set_ylim(0, 10)
ax.set_ylabel("Score (0-10)")
ax.set_title(f"Quarterly score trend — {focus_hotel}")
ax.legend(loc="lower left", fontsize=8, ncol=3)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "dashboard_quarterly_trend.png", dpi=110)
plt.show()

# --- Leaderboard ---
_board = sorted(dashboard_data["leaderboard"], key=lambda r: r["rank"], reverse=True)
_names = [f"#{r['rank']} {r['Hotel_Name']}" for r in _board]
_lb_scores = [r["overall_score"] for r in _board]
_colors = ["#2563eb" if r["Hotel_Name"] == focus_hotel else "#cbd5e1" for r in _board]

fig, ax = plt.subplots(figsize=(7, 0.6 * len(_board) + 1))
ax.barh(_names, _lb_scores, color=_colors)
ax.set_xlim(0, 10)
ax.set_xlabel("Overall score (mean Reviewer_Score)")
ax.set_title("Leaderboard")
for i, score in enumerate(_lb_scores):
    ax.text(score + 0.1, i, f"{score:.2f}", va="center", fontsize=9)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "dashboard_leaderboard.png", dpi=110)
plt.show()

# --- Competitor gaps ---
if dashboard_data["competitor_gaps"]:
    _gaps = dashboard_data["competitor_gaps"]
    fig, axes = plt.subplots(len(_gaps), 1, figsize=(8, 3 * len(_gaps)), squeeze=False)
    for ax, comp in zip(axes[:, 0], _gaps):
        cats = [c for c in CHART_ASPECTS if comp.get(c) is not None]
        vals = [comp[c] for c in cats]
        colors = ["#2563eb" if v >= 0 else "#ef4444" for v in vals]
        ax.barh(cats, vals, color=colors)
        ax.axvline(0, color="#94a3b8", linewidth=1)
        ax.set_title(f"Gap vs. {comp['competitor']} (positive = focus hotel ahead)")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "dashboard_competitor_gaps.png", dpi=110)
    plt.show()
else:
    print("No competitors in this dataset yet — skipping gap chart.")

print(f"\nFigures saved to {FIGURES_DIR}")


## Notes

- **If Colab disconnects** (idle timeout or 12h session cap): reconnect, re-run cells 1-8 to
  reload the models, then re-run cell 10 — it picks up exactly where it left off.
- **If you hit `CUDA out of memory`**: lower `OVERALL_BATCH_SIZE` / `ABSA_BATCH_SIZE` in cell 6
  (try halving them) and re-run from cell 6 onward.
- The final `outputs/reviews_enriched.parquet` in Drive is exactly the file
  `src/nlp_pipeline.py` would have produced — download it and drop it into the project's
  `outputs/` folder to continue with Phase 3 (feature engineering).
